# Star Cluster Injection Pipeline -- Actual Rubin CoaddPsf

This notebook performs the full injection-detection-completeness pipeline
using the **actual spatially-varying CoaddPsf** from the Rubin deepCoadd
exposure, queried at each cluster position in focal-plane coordinates.

Cluster profiles are generated with `src/light_profiles.py` (King, Plummer,
EFF, Sersic). GalSim is used **only** for PSF convolution via
`galsim.InterpolatedImage` + `galsim.Convolve`.

## Pipeline Flow

```
1. Load deepCoadd from Butler
2. PSF diagnostic -- measure FWHM across the image, validate coordinates
3. Configure injection parameters
4. Generate injection catalog
5. Inject clusters (light_profiles + actual CoaddPsf convolution)
6. Injection diagnostic plots
7. Detection (user-supplied)
8. Retrieval and completeness curves
```

---
## 1. Imports

In [ ]:
import os
os.chdir(os.path.expanduser('~'))
import sys
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

pipeline_path = os.path.expanduser('~/WORK/INJECT/star-cluster-injection-pipeline')
if pipeline_path not in sys.path:
    sys.path.insert(0, pipeline_path)

import galsim
from lsst.daf.butler import Butler
from lsst.geom import Point2D

from src.config import InjectionConfig, ClusterConfig
from src.pipeline import InjectionPipeline
from src.light_profiles import (
    KingProfile,
    PlummerProfile,
    EFFProfile,
    SersicProfile,
    mag_to_flux,
)
from src.retrieval import ClusterRetrieval

plt.style.use('tableau-colorblind10')
%matplotlib inline

print('All imports successful.')

---
## 2. Load deepCoadd from Butler

In [ ]:
butler = Butler('dp02', collections='2.2i/runs/DP0.2')

data_id = {'tract': 3828, 'patch': 24, 'band': 'i'}
coadd = butler.get('deepCoadd', dataId=data_id)

CUTOUT = 1500  # pixels -- increase for production runs
image_full = coadd.image.array.copy()
image = image_full[:CUTOUT, :CUTOUT].copy()

print(f'Loaded deepCoadd: tract={data_id["tract"]}  patch={data_id["patch"]}  band={data_id["band"]}')
print(f'Full coadd shape : {image_full.shape}')
print(f'Cutout shape     : {image.shape}')
print(f'Image dtype      : {image.dtype}')
print(f'Image min        : {image.min():.4f}')
print(f'Image max        : {image.max():.4f}')
print(f'Image median     : {np.median(image):.4f}')

fig, ax = plt.subplots(figsize=(7, 7))
vmin, vmax = np.percentile(image, [1, 99])
ax.imshow(image, cmap='gray', origin='lower', vmin=vmin, vmax=vmax)
ax.set_title('Input deepCoadd cutout')
plt.show()

---
## 3. PSF Diagnostic and Validation

The CoaddPsf must be queried in **focal plane coordinates**, not cutout
pixel coordinates. The bounding box offset is retrieved from the Butler
exposure and added to every position query.

A deepCoadd cutout does NOT start at pixel (0, 0) in focal plane
coordinates. Passing raw cutout positions to `computeImage()` will raise
`InvalidPsfError` because those coordinates fall outside the range of
any input exposure that went into the coadd.

In [ ]:
# ---- constants ---------------------------------------------------------------
PIXEL_SCALE = 0.2    # arcsec per pixel (Rubin standard)
ZERO_POINT  = 27.0   # AB magnitude zero point

# ---- bounding box offset -----------------------------------------------------
psf_obj    = coadd.getPsf()
bbox       = coadd.getBBox()
BBOX_X_MIN = bbox.getMinX()
BBOX_Y_MIN = bbox.getMinY()
x_max      = bbox.getMaxX()
y_max      = bbox.getMaxY()
x_mid      = (BBOX_X_MIN + x_max) // 2
y_mid      = (BBOX_Y_MIN + y_max) // 2

print(f'Coadd bounding box : x=[{BBOX_X_MIN}, {x_max}]  y=[{BBOX_Y_MIN}, {y_max}]')
print(f'Coadd centre       : ({x_mid}, {y_mid})')
print(f'Bbox offset stored : BBOX_X_MIN={BBOX_X_MIN}, BBOX_Y_MIN={BBOX_Y_MIN}')
print()

# ---- sample named positions in focal plane coords ----------------------------
np.random.seed(99)
test_positions = {
    'centre':       (x_mid,        y_mid),
    'top-left':     (BBOX_X_MIN + 50,   y_max - 50),
    'top-right':    (x_max - 50,        y_max - 50),
    'bottom-left':  (BBOX_X_MIN + 50,   BBOX_Y_MIN + 50),
    'bottom-right': (x_max - 50,        BBOX_Y_MIN + 50),
    'random-1':     (np.random.randint(BBOX_X_MIN+50, x_max-50),
                     np.random.randint(BBOX_Y_MIN+50, y_max-50)),
    'random-2':     (np.random.randint(BBOX_X_MIN+50, x_max-50),
                     np.random.randint(BBOX_Y_MIN+50, y_max-50)),
}

print('=== PSF Sampling Diagnostic ===')
print(f'{"Position":<15} {"x":>6} {"y":>6} {"FWHM (px)":>10} {"FWHM (arcsec)":>14}  Status')
print('-' * 68)

psf_results = {}
for name, (x, y) in test_positions.items():
    try:
        point    = Point2D(x, y)
        shape    = psf_obj.computeShape(point)
        fwhm_px  = shape.getDeterminantRadius() * 2.355
        fwhm_as  = fwhm_px * PIXEL_SCALE
        status   = ' OK' if 1.5 < fwhm_px < 8.0 else '  SUSPICIOUS'
        psf_results[name] = {'x': x, 'y': y, 'fwhm_px': fwhm_px, 'fwhm_as': fwhm_as}
        print(f'{name:<15} {x:>6} {y:>6} {fwhm_px:>10.3f} {fwhm_as:>14.3f}  {status}')
    except Exception as e:
        print(f'{name:<15} {x:>6} {y:>6}  {"ERROR":>10}  --  {str(e).splitlines()[0]}')
        psf_results[name] = None

valid_fwhms = [v['fwhm_px'] for v in psf_results.values() if v is not None]
if valid_fwhms:
    PSF_FWHM = float(np.median(valid_fwhms))
    print()
    print(f'Median PSF FWHM : {PSF_FWHM:.3f} px  ({PSF_FWHM * PIXEL_SCALE:.3f} arcsec)')
    print(f'PSF_FWHM set to {PSF_FWHM:.3f} px (used as Gaussian fallback only)')
else:
    PSF_FWHM = 3.5
    print()
    print('ALL positions failed -- using default PSF_FWHM = 3.5 px')
    print('Check bounding box output above and test manually')

In [ ]:
# ---- Full grid measurement ---------------------------------------------------
N_GRID = 5
xs = np.linspace(BBOX_X_MIN + 100, x_max - 100, N_GRID).astype(int)
ys = np.linspace(BBOX_Y_MIN + 100, y_max - 100, N_GRID).astype(int)

grid_fwhm_px = np.full((N_GRID, N_GRID), np.nan)
grid_fwhm_as = np.full((N_GRID, N_GRID), np.nan)

print(f'Measuring PSF on {N_GRID}x{N_GRID} grid ...')
for i, y in enumerate(ys):
    for j, x in enumerate(xs):
        try:
            shape = psf_obj.computeShape(Point2D(int(x), int(y)))
            grid_fwhm_px[i, j] = shape.getDeterminantRadius() * 2.355
            grid_fwhm_as[i, j] = grid_fwhm_px[i, j] * PIXEL_SCALE
        except Exception:
            pass

valid = grid_fwhm_px[~np.isnan(grid_fwhm_px)]
if len(valid) > 0:
    PSF_FWHM = float(np.median(valid))
    print(f'Grid PSF statistics ({N_GRID}x{N_GRID} = {N_GRID*N_GRID} positions):')
    print(f'  Median : {PSF_FWHM:.4f} px  ({PSF_FWHM * PIXEL_SCALE:.4f} arcsec)')
    print(f'  Min    : {valid.min():.4f} px')
    print(f'  Max    : {valid.max():.4f} px')
    print(f'  Std    : {valid.std():.4f} px  (spatial variation)')
    print(f'  N good : {len(valid)} / {N_GRID*N_GRID}')
else:
    print('No valid grid measurements.')

print()
print(f'PSF_FWHM = {PSF_FWHM:.3f} px (Gaussian fallback value)')

In [ ]:
# ---- PSF diagnostic plots ----------------------------------------------------
fig = plt.figure(figsize=(16, 12))
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.35, wspace=0.35)

# Panel 1: FWHM spatial map (pixels)
ax1 = fig.add_subplot(gs[0, 0])
im1 = ax1.pcolormesh(
    xs - BBOX_X_MIN, ys - BBOX_Y_MIN, grid_fwhm_px,
    cmap='RdYlGn_r', vmin=valid.min() - 0.1, vmax=valid.max() + 0.1
)
plt.colorbar(im1, ax=ax1, label='FWHM (px)')
for i in range(N_GRID):
    for j in range(N_GRID):
        if not np.isnan(grid_fwhm_px[i, j]):
            ax1.text(
                xs[j] - BBOX_X_MIN, ys[i] - BBOX_Y_MIN,
                f'{grid_fwhm_px[i,j]:.2f}',
                ha='center', va='center', fontsize=7,
            )
ax1.set_xlabel('Cutout x (pixels)')
ax1.set_ylabel('Cutout y (pixels)')
ax1.set_title('PSF FWHM Spatial Map (pixels)')

# Panel 2: FWHM spatial map (arcsec)
ax2 = fig.add_subplot(gs[0, 1])
im2 = ax2.pcolormesh(
    xs - BBOX_X_MIN, ys - BBOX_Y_MIN, grid_fwhm_as,
    cmap='RdYlGn_r',
)
plt.colorbar(im2, ax=ax2, label='FWHM (arcsec)')
ax2.set_xlabel('Cutout x (pixels)')
ax2.set_ylabel('Cutout y (pixels)')
ax2.set_title('PSF FWHM Spatial Map (arcsec)')

# Panel 3: FWHM histogram
ax3 = fig.add_subplot(gs[0, 2])
ax3.hist(valid, bins=max(5, len(valid)//2), color='steelblue',
         edgecolor='white', linewidth=0.8)
ax3.axvline(PSF_FWHM, color='tomato', ls='--', lw=1.5,
            label=f'Median = {PSF_FWHM:.3f} px')
ax3.axvline(valid.mean(), color='darkorange', ls=':', lw=1.5,
            label=f'Mean   = {valid.mean():.3f} px')
ax3.set_xlabel('FWHM (pixels)')
ax3.set_ylabel('Count')
ax3.set_title('PSF FWHM Distribution')
ax3.legend(fontsize=9)

# Panel 4: PSF image at centre
ax4 = fig.add_subplot(gs[1, 0])
if psf_results.get('centre') is not None:
    psf_centre_img = psf_obj.computeImage(Point2D(x_mid, y_mid))
    psf_centre_arr = psf_centre_img.array
    im4 = ax4.imshow(psf_centre_arr, origin='lower', cmap='magma')
    plt.colorbar(im4, ax=ax4, label='Normalised flux')
    ax4.set_title(
        f'PSF Image at Centre\n'
        f'FWHM = {psf_results["centre"]["fwhm_px"]:.3f} px'
    )
else:
    ax4.text(0.5, 0.5, 'PSF at centre\nfailed', ha='center', va='center',
             transform=ax4.transAxes)
ax4.set_xlabel('x (pixels)')
ax4.set_ylabel('y (pixels)')

# Panel 5: PSF radial profile
ax5 = fig.add_subplot(gs[1, 1])
if psf_results.get('centre') is not None:
    cy_p, cx_p = np.array(psf_centre_arr.shape) // 2
    y_idx, x_idx = np.indices(psf_centre_arr.shape)
    r = np.sqrt((x_idx - cx_p)**2 + (y_idx - cy_p)**2).ravel()
    flux_r = psf_centre_arr.ravel()
    r_sort = np.argsort(r)

    ax5.scatter(r[r_sort], flux_r[r_sort], s=4, alpha=0.4,
                color='steelblue', label='All pixels')

    r_bins = np.arange(0, r.max(), 0.5)
    r_centres = 0.5 * (r_bins[:-1] + r_bins[1:])
    r_median = [
        np.median(flux_r[(r >= r_bins[k]) & (r < r_bins[k+1])])
        for k in range(len(r_bins)-1)
    ]
    ax5.plot(r_centres, r_median, color='tomato', lw=2, label='Binned median')
    ax5.axvline(
        psf_results['centre']['fwhm_px'] / 2, ls='--',
        color='darkorange', lw=1.5,
        label=f'Half-FWHM = {psf_results["centre"]["fwhm_px"]/2:.2f} px'
    )
    ax5.legend(fontsize=8)
ax5.set_xlabel('Radius (pixels)')
ax5.set_ylabel('Normalised flux')
ax5.set_title('PSF Radial Profile at Centre')

# Panel 6: FWHM variation along diagonal
ax6 = fig.add_subplot(gs[1, 2])
diag_fwhm = [grid_fwhm_px[i, i] for i in range(N_GRID)
             if not np.isnan(grid_fwhm_px[i, i])]
diag_pos = [
    np.sqrt((xs[i] - BBOX_X_MIN)**2 + (ys[i] - BBOX_Y_MIN)**2)
    for i in range(N_GRID)
    if not np.isnan(grid_fwhm_px[i, i])
]
if diag_fwhm:
    ax6.plot(diag_pos, diag_fwhm, 'o-', color='steelblue',
             lw=2, ms=6, label='FWHM along diagonal')
    ax6.axhline(PSF_FWHM, ls='--', color='tomato', lw=1.5,
                label=f'Median = {PSF_FWHM:.3f} px')
    ax6.fill_between(
        [min(diag_pos), max(diag_pos)],
        PSF_FWHM - valid.std(), PSF_FWHM + valid.std(),
        alpha=0.2, color='tomato', label='1 sigma range'
    )
    ax6.legend(fontsize=8)
ax6.set_xlabel('Distance from corner (pixels)')
ax6.set_ylabel('FWHM (pixels)')
ax6.set_title('PSF Spatial Variation Along Diagonal')

fig.suptitle(
    f'CoaddPsf Diagnostic -- tract {data_id["tract"]}  '
    f'patch {data_id["patch"]}  {data_id["band"]}-band\n'
    f'Median FWHM = {PSF_FWHM:.3f} px ({PSF_FWHM*PIXEL_SCALE:.3f} arcsec)  '
    f'Spatial variation = {valid.std():.4f} px',
    fontsize=13,
)
plt.tight_layout()
plt.show()
print('PSF diagnostic complete.')

---
## 4. Injection Configuration

In [ ]:
cluster_cfg = ClusterConfig(
    profile_type      = 'king',
    method            = 'smooth',
    mag_min           = 20.0,
    mag_max           = 26.0,
    r_half_min        = 2.0,
    r_half_max        = 10.0,
    concentration_min = 5.0,
    concentration_max = 30.0,
)

config = InjectionConfig(
    run_name           = 'rubin_psf_injection',
    n_clusters         = 100,
    seed               = 42,
    edge_buffer        = 50,
    add_noise          = True,
    use_actual_psf     = True,
    save_injected_image= False,
    cluster_config     = cluster_cfg,
    tract              = data_id['tract'],
    patch              = data_id['patch'],
    band               = data_id['band'],
    output_dir         = 'outputs/rubin_psf_injection',
)

os.makedirs(config.output_dir, exist_ok=True)

print(config)
print()
print(f'PSF_FWHM (measured, Gaussian fallback) : {PSF_FWHM:.3f} px')
print(f'PIXEL_SCALE                            : {PIXEL_SCALE} arcsec/px')
print(f'ZERO_POINT                             : {ZERO_POINT}')
print(f'BBOX_X_MIN                             : {BBOX_X_MIN}')
print(f'BBOX_Y_MIN                             : {BBOX_Y_MIN}')

---
## 5. Generate Injection Catalog

In [ ]:
pipe = InjectionPipeline(config)
pipe.load_data(image=image)
catalog = pipe.generate_catalog()

print(f'Generated catalog with {len(catalog)} entries.')
print()
print(f'{"id":>4}  {"x":>7}  {"y":>7}  {"mag":>6}  {"r_half":>7}  {"conc":>6}')
print('-' * 48)
for entry in catalog[:10]:
    print(
        f'{entry["id"]:>4}  '
        f'{entry["x"]:>7.1f}  '
        f'{entry["y"]:>7.1f}  '
        f'{entry["magnitude"]:>6.2f}  '
        f'{entry["r_half"]:>7.2f}  '
        f'{entry.get("concentration", 0):>6.1f}'
    )
print('  ... (first 10 shown)')

---
## 6. Define Injection Functions

### 6a. `make_profile_image` -- generate cluster stamp from `light_profiles.py`

This uses the physically-motivated profile classes (King, Plummer, EFF,
Sersic) from `src/light_profiles.py`. The output is a 2D array normalised
to unit sum.

### 6b. `get_actual_psf` -- fetch the CoaddPsf at a specific position

Converts cutout pixel coordinates to focal-plane coordinates using the
bounding box offset, then calls `computeImage()` and wraps the result
as a `galsim.InterpolatedImage`.

### 6c. `inject_clusters_rubin_psf` -- the main injection loop

In [ ]:
def make_profile_image(entry, pixel_scale=PIXEL_SCALE):
    """
    Generate the intrinsic 2-D cluster surface-brightness stamp using
    src/light_profiles.py.

    Parameters
    ----------
    entry : dict
        One row from the injection catalog. Expected keys:
        'profile_type', 'r_half' (pixels), 'magnitude',
        'concentration' (King c or EFF gamma or Sersic n).
    pixel_scale : float
        Arcsec per pixel (kept for interface consistency).

    Returns
    -------
    image_2d : np.ndarray (stamp_size x stamp_size, float64)
        Normalised surface-brightness map (sums to 1).
    stamp_size : int
        Side length of the returned stamp (always odd).
    """
    profile_type = entry.get('profile_type', 'plummer').lower()
    r_half       = float(entry.get('r_half', 5.0))
    magnitude    = float(entry.get('magnitude', 22.0))
    age          = float(entry.get('age_gyr', 1.0))
    conc         = float(entry.get('concentration', 10.0))

    # Stamp large enough to capture the full profile
    stamp_size = max(51, int(10 * r_half))
    if stamp_size % 2 == 0:
        stamp_size += 1

    shape = (stamp_size, stamp_size)

    if profile_type == 'king':
        prof = KingProfile(
            r_half=r_half, concentration=conc, age=age,
            magnitude=magnitude, zero_point=ZERO_POINT,
        )
    elif profile_type == 'plummer':
        prof = PlummerProfile(
            r_half=r_half, age=age,
            magnitude=magnitude, zero_point=ZERO_POINT,
        )
    elif profile_type == 'eff':
        gamma = conc if conc > 2.01 else 2.5
        prof = EFFProfile(
            r_half=r_half, gamma=gamma, age=age,
            magnitude=magnitude, zero_point=ZERO_POINT,
        )
    elif profile_type == 'sersic':
        n = conc if conc >= 0.3 else 1.0
        prof = SersicProfile(
            r_half=r_half, sersic_n=n, age=age,
            magnitude=magnitude, zero_point=ZERO_POINT,
        )
    else:
        raise ValueError(f'Unknown profile_type: "{profile_type}". '
                         f'Choose from king, plummer, eff, sersic.')

    image_2d = prof.generate_2d(shape).astype(np.float64)

    total = image_2d.sum()
    if total > 0:
        image_2d /= total

    return image_2d, stamp_size


# Sanity check
_test_arr, _test_sz = make_profile_image(catalog[0])
print(f'make_profile_image check: shape={_test_arr.shape}, '
      f'sum={_test_arr.sum():.6f}, stamp_size={_test_sz}')

In [ ]:
def get_actual_psf(psf_obj, cutout_x, cutout_y,
                   bbox_x_min, bbox_y_min, pixel_scale):
    """
    Fetch the actual CoaddPsf at a cutout pixel position and return
    it as a GalSim InterpolatedImage.

    Converts cutout coords -> focal plane coords using the bbox offset.

    Parameters
    ----------
    psf_obj    : lsst.meas.algorithms.CoaddPsf
    cutout_x   : float -- x in cutout pixel coordinates (0-indexed)
    cutout_y   : float -- y in cutout pixel coordinates (0-indexed)
    bbox_x_min : int   -- coadd bounding box x offset
    bbox_y_min : int   -- coadd bounding box y offset
    pixel_scale: float -- arcsec per pixel

    Returns
    -------
    psf_gs  : galsim.InterpolatedImage
    fwhm_px : float -- PSF FWHM at this position in pixels
    """
    focal_x = cutout_x + bbox_x_min
    focal_y = cutout_y + bbox_y_min
    point   = Point2D(focal_x, focal_y)

    # Get the PSF kernel image at this position
    psf_image = psf_obj.computeImage(point)
    psf_array = psf_image.array.astype(np.float64)

    # Normalise to unit sum
    psf_sum = psf_array.sum()
    if psf_sum > 0:
        psf_array /= psf_sum

    # Wrap as GalSim object
    gs_img = galsim.Image(psf_array, scale=pixel_scale)
    psf_gs = galsim.InterpolatedImage(gs_img, normalization='flux')

    # Measure FWHM at this position
    shape   = psf_obj.computeShape(point)
    fwhm_px = shape.getDeterminantRadius() * 2.355

    return psf_gs, fwhm_px


# Sanity check -- fetch PSF at the centre of our cutout
_test_cx = image.shape[1] // 2
_test_cy = image.shape[0] // 2
try:
    _psf_test, _fwhm_test = get_actual_psf(
        psf_obj, _test_cx, _test_cy, BBOX_X_MIN, BBOX_Y_MIN, PIXEL_SCALE
    )
    print(f'get_actual_psf check: cutout ({_test_cx},{_test_cy}) -> '
          f'focal ({_test_cx+BBOX_X_MIN},{_test_cy+BBOX_Y_MIN}) -> '
          f'FWHM = {_fwhm_test:.3f} px -- OK')
except Exception as e:
    print(f'get_actual_psf check FAILED: {e}')
    print('The injection will fall back to Gaussian PSF at failing positions.')

In [ ]:
def inject_clusters_rubin_psf(image, catalog,
                               psf_obj,
                               bbox_x_min,
                               bbox_y_min,
                               psf_fwhm_fallback,
                               pixel_scale  = PIXEL_SCALE,
                               add_noise    = True,
                               rng_seed     = 42):
    """
    Inject clusters using the actual Rubin CoaddPsf.

    For each cluster:
      1. make_profile_image()  ->  2D stamp from light_profiles.py
      2. get_actual_psf()      ->  CoaddPsf kernel at that position
      3. galsim.Convolve()     ->  PSF-convolved stamp
      4. Scale to correct flux and inject into image

    Falls back to a Gaussian PSF if the actual PSF fetch fails at any
    position (e.g. near edges outside the coadd footprint).

    Parameters
    ----------
    image              : 2D ndarray
    catalog            : list[dict]
    psf_obj            : lsst PSF from coadd.getPsf()
    bbox_x_min         : int -- bounding box x offset
    bbox_y_min         : int -- bounding box y offset
    psf_fwhm_fallback  : float -- Gaussian FWHM in pixels (fallback)
    pixel_scale        : float -- arcsec per pixel
    add_noise          : bool
    rng_seed           : int

    Returns
    -------
    injected_image : 2D ndarray
    injection_info : list[dict]
    """
    ny, nx         = image.shape
    injected       = image.copy().astype(np.float64)
    rng_np         = np.random.default_rng(rng_seed)
    injection_info = []

    # Gaussian fallback PSF
    gaussian_psf = galsim.Gaussian(fwhm=psf_fwhm_fallback * pixel_scale)

    n_ok           = 0
    n_failed       = 0
    n_psf_fallback = 0

    print(f'  PSF mode          : actual Rubin CoaddPsf (Gaussian fallback FWHM={psf_fwhm_fallback:.2f} px)')
    print(f'  Bbox offset       : ({bbox_x_min}, {bbox_y_min})')
    print(f'  N clusters        : {len(catalog)}')
    print()

    for i, entry in enumerate(catalog):
        try:
            cx = int(round(entry['x']))
            cy = int(round(entry['y']))

            # ---- Step 1: cluster profile from light_profiles.py ----
            profile_arr, stamp_size = make_profile_image(entry, pixel_scale)

            # ---- Step 2: wrap as GalSim InterpolatedImage ----
            gs_cluster_img = galsim.Image(profile_arr, scale=pixel_scale)
            cluster_gs     = galsim.InterpolatedImage(
                gs_cluster_img, normalization='flux'
            )

            # ---- Step 3: get the actual PSF at this position ----
            try:
                psf_here, fwhm_here = get_actual_psf(
                    psf_obj, cx, cy, bbox_x_min, bbox_y_min, pixel_scale
                )
            except Exception as psf_err:
                psf_here  = gaussian_psf
                fwhm_here = psf_fwhm_fallback
                n_psf_fallback += 1
                if n_psf_fallback <= 5:
                    print(f'  PSF fallback at cutout ({cx},{cy}): '
                          f'{str(psf_err).splitlines()[0]}')

            # ---- Step 4: convolve ----
            convolved = galsim.Convolve([cluster_gs, psf_here])

            # ---- Step 5: draw stamp ----
            out_img = galsim.Image(stamp_size, stamp_size, scale=pixel_scale)
            convolved.drawImage(image=out_img, method='no_pixel')
            stamp = out_img.array.copy().astype(np.float64)

            # ---- Step 6: scale to correct total flux ----
            total_flux = mag_to_flux(entry['magnitude'], zero_point=ZERO_POINT)
            stamp_sum  = stamp.sum()
            if stamp_sum > 0:
                stamp *= total_flux / stamp_sum

            # ---- Step 7: Poisson-like noise ----
            if add_noise:
                noise_sigma = np.sqrt(np.clip(stamp, 0, None))
                stamp += rng_np.normal(
                    0.0, np.where(noise_sigma > 0, noise_sigma, 1e-10)
                )

            # ---- Step 8: inject into image with boundary clipping ----
            sh, sw = stamp.shape
            y0 = cy - sh // 2;  y1 = y0 + sh
            x0 = cx - sw // 2;  x1 = x0 + sw

            iy0 = max(y0, 0);  iy1 = min(y1, ny)
            ix0 = max(x0, 0);  ix1 = min(x1, nx)

            if iy0 >= iy1 or ix0 >= ix1:
                continue

            sy0 = iy0 - y0;  sy1 = sy0 + (iy1 - iy0)
            sx0 = ix0 - x0;  sx1 = sx0 + (ix1 - ix0)

            injected[iy0:iy1, ix0:ix1] += stamp[sy0:sy1, sx0:sx1]

            info                = dict(entry)
            info['stamp']       = stamp
            info['stamp_flux']  = float(stamp.sum())
            info['total_flux']  = total_flux
            info['psf_fwhm_px'] = fwhm_here
            injection_info.append(info)
            n_ok += 1

        except Exception as exc:
            n_failed += 1
            if n_failed <= 10:
                print(f'  Cluster {i} (id={entry.get("id", "?")}) failed: {exc}')

    if n_psf_fallback > 5:
        print(f'  ... ({n_psf_fallback} total PSF fallbacks, only first 5 shown)')
    if n_failed > 10:
        print(f'  ... ({n_failed} total failures, only first 10 shown)')

    print()
    print(f'Injection complete.')
    print(f'  Successful        : {n_ok}')
    print(f'  Failed            : {n_failed}')
    print(f'  PSF fallback used : {n_psf_fallback}')

    return injected.astype(image.dtype), injection_info


print('Injection functions defined.')

---
## 7. Run Injection

In [ ]:
print(f'Injecting {len(catalog)} clusters ...')
print(f'  Profile : {config.cluster_config.profile_type}')
print(f'  PSF     : actual Rubin CoaddPsf')
print()

injected_image, injection_info = inject_clusters_rubin_psf(
    image,
    catalog,
    psf_obj           = psf_obj,
    bbox_x_min        = BBOX_X_MIN,
    bbox_y_min        = BBOX_Y_MIN,
    psf_fwhm_fallback = PSF_FWHM,
    pixel_scale       = PIXEL_SCALE,
    add_noise         = config.add_noise,
    rng_seed          = config.seed,
)

print()
print(f'Successfully injected {len(injection_info)} / {len(catalog)} clusters.')

---
## 8. Injection Diagnostic Plots

In [ ]:
# ---- Before / After / Difference ---------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

vmin, vmax = np.percentile(image, [1, 99])
kw = dict(cmap='gray', origin='lower', vmin=vmin, vmax=vmax)

axes[0].imshow(image, **kw)
axes[0].set_title('Original')

axes[1].imshow(injected_image, **kw)
axes[1].set_title('Injected')

diff = injected_image.astype(np.float64) - image.astype(np.float64)
dv = np.percentile(np.abs(diff), 99)
axes[2].imshow(diff, cmap='RdBu_r', origin='lower', vmin=-dv, vmax=dv)
axes[2].set_title('Difference')

for ax in axes:
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
plt.savefig(os.path.join(config.output_dir, 'before_after.png'), dpi=150)
plt.show()

In [ ]:
# ---- Stamp grid --------------------------------------------------------------
n_show = min(20, len(injection_info))
n_cols = 5
n_rows = int(np.ceil(n_show / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 3 * n_rows))

for idx, ax in enumerate(axes.flat):
    if idx < n_show:
        info  = injection_info[idx]
        stamp = info['stamp']
        sv1   = np.percentile(stamp, 1)
        sv2   = np.percentile(stamp, 99)
        ax.imshow(stamp, origin='lower', cmap='magma', vmin=sv1, vmax=sv2)
        ax.set_title(
            f'm={info["magnitude"]:.1f}  r={info["r_half"]:.1f}px\n'
            f'psf={info["psf_fwhm_px"]:.2f}px',
            fontsize=6.5,
        )
    ax.set_xticks([])
    ax.set_yticks([])

plt.suptitle(
    f'Injected Cluster Stamps (actual CoaddPsf convolved) -- '
    f'{config.cluster_config.profile_type.upper()} profile',
    fontsize=13,
)
plt.tight_layout()
plt.savefig(os.path.join(config.output_dir, 'stamp_grid.png'), dpi=150)
plt.show()

In [ ]:
# ---- PSF FWHM distribution at injection positions ----------------------------
inj_fwhms = [info['psf_fwhm_px'] for info in injection_info]

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(inj_fwhms, bins=20, color='steelblue', edgecolor='white', lw=0.8)
ax.axvline(np.median(inj_fwhms), ls='--', color='tomato', lw=1.5,
           label=f'Median = {np.median(inj_fwhms):.3f} px')
ax.set_xlabel('PSF FWHM at injection position (pixels)')
ax.set_ylabel('Count')
ax.set_title('Distribution of PSF FWHM Across Injection Positions')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(config.output_dir, 'injection_psf_fwhm.png'), dpi=150)
plt.show()

In [ ]:
# ---- Injected parameter distributions ----------------------------------------
mags  = [info['magnitude']                  for info in injection_info]
rhs   = [info['r_half']                     for info in injection_info]
concs = [info.get('concentration', np.nan)  for info in injection_info]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(mags,  bins=20, color='steelblue',  edgecolor='k')
axes[0].set_xlabel('Magnitude');           axes[0].set_ylabel('N')

axes[1].hist(rhs,   bins=20, color='darkorange', edgecolor='k')
axes[1].set_xlabel('r_half (pixels)')

axes[2].hist(concs, bins=20, color='seagreen',   edgecolor='k')
axes[2].set_xlabel('Concentration')

plt.suptitle('Injected Cluster Parameters', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(config.output_dir, 'injected_params.png'), dpi=150)
plt.show()

---
## 9. Detection

Replace this cell with your detection pipeline. The only requirement is
that `detections` is a list of dicts with at minimum keys `'x'` and `'y'`.

### Required detection output format

```python
detections = [
    {
        'x':          142.3,   # REQUIRED -- detected centroid x (pixels)
        'y':          876.1,   # REQUIRED -- detected centroid y (pixels)
        'flux':      1234.5,   # optional -- measured total flux
        'magnitude':   22.3,   # optional -- measured apparent magnitude
        'r_half':       4.2,   # optional -- measured half-light radius (pixels)
        'snr':          8.4,   # optional -- peak signal-to-noise ratio
        'flag':           0,   # optional -- 0=good, anything else=bad
    },
    # ... one dict per detected source
]
```

| Key | Unlocks |
|---|---|
| `x`, `y` | Spatial matching to injections |
| `magnitude` | Completeness vs magnitude curve |
| `r_half` | Completeness vs half-light radius curve |
| `magnitude` + `r_half` | 2D completeness map |
| `snr` | Completeness vs SNR curve |
| `flux` | Flux recovery / photometric offset plots |

In [ ]:
# ---- Detection: use the same MCI pipeline method as your other notebooks -----
from src.detection import run_cluster_detection, plot_mci_diagram

# Match the MCI-style settings you have used in your pipeline notebooks
USE_MCI = True
MCI_MAX = 0.85
THRESHOLD_SIGMA = 3.0
SNR_MIN = 3.0
R_HALF_MIN = 1.0
ELLIPTICITY_MAX = 0.6
USE_MULTISCALE = True

# Build detection scales from PSF if available
_det_cfg = getattr(config, 'detection_config', None)
_psf_fwhm = float(getattr(_det_cfg, 'psf_fwhm', 3.5))
_pixel_scale = float(globals().get('PIXEL_SCALE', 0.2))
R_HALF_SCALES = [max(2.0, _psf_fwhm / 2), _psf_fwhm, _psf_fwhm * 2, _psf_fwhm * 4]

detections = run_cluster_detection(
    image=injected_image,
    psf_fwhm=_psf_fwhm,
    threshold_sigma=THRESHOLD_SIGMA,
    r_half_scales=R_HALF_SCALES,
    mci_max=MCI_MAX,
    snr_min=SNR_MIN,
    r_half_min=R_HALF_MIN,
    ellipticity_max=ELLIPTICITY_MAX,
    box_size=64,
    pixel_scale=_pixel_scale,
    use_multiscale=USE_MULTISCALE,
    use_mci=USE_MCI,
    verbose=True,
)

print(f'MCI detections: {len(detections)}')

# Optional MCI diagnostic plot (safe if MCI keys are present)
if USE_MCI and len(detections) > 0 and 'inner_mci' in detections[0]:
    _ = plot_mci_diagram(detections, injection_info=injection_info)
    plt.show()

In [ ]:
# ---- Optional: tune MCI parameters and re-run detection ----------------------
# Use this if you want quick sensitivity tests for sample completeness plots.

MCI_MAX = 0.85
THRESHOLD_SIGMA = 3.0
SNR_MIN = 3.0
R_HALF_MIN = 1.0
ELLIPTICITY_MAX = 0.6
USE_MULTISCALE = True

_det_cfg = getattr(config, 'detection_config', None)
_psf_fwhm = float(getattr(_det_cfg, 'psf_fwhm', 3.5))
_pixel_scale = float(globals().get('PIXEL_SCALE', 0.2))
R_HALF_SCALES = [max(2.0, _psf_fwhm / 2), _psf_fwhm, _psf_fwhm * 2, _psf_fwhm * 4]

detections = run_cluster_detection(
    image=injected_image,
    psf_fwhm=_psf_fwhm,
    threshold_sigma=THRESHOLD_SIGMA,
    r_half_scales=R_HALF_SCALES,
    mci_max=MCI_MAX,
    snr_min=SNR_MIN,
    r_half_min=R_HALF_MIN,
    ellipticity_max=ELLIPTICITY_MAX,
    box_size=64,
    pixel_scale=_pixel_scale,
    use_multiscale=USE_MULTISCALE,
    use_mci=True,
    verbose=True,
)

print('Re-ran MCI detection with current tuning parameters.')
print(f'MCI detections: {len(detections)}')

### MCI Method Aligned With Your Pipeline

Detection now uses the same method used in your other notebooks:
`run_cluster_detection(..., use_mci=True, mci_max=0.85)`.

Cell 26 runs it once. Cell 27 lets you tune MCI parameters and re-run before retrieval/completeness.

---
## 10. Retrieval and Completeness

These cells produce meaningful results once the detection cell above
is populated with real detections.

In [ ]:
MATCH_RADIUS = 5.0  # pixels

if len(detections) == 0:
    print('No detections yet -- populate the detection cell above first.')
    print('Skipping retrieval and completeness.')
else:
    retrieval = ClusterRetrieval(injection_info, detections)
    retrieval.match_detections(match_radius=MATCH_RADIUS)
    stats = retrieval.get_summary_statistics()

    print('=== Retrieval Summary ===')
    print(f'  Injected         : {stats["n_injected"]}')
    print(f'  Recovered        : {stats["n_detected"]}')
    print(f'  Missed           : {stats["n_injected"] - stats["n_detected"]}')
    print(f'  Completeness     : {stats["overall_completeness"]:.1%}')
    print(f'  50% mag limit    : {stats["mag_50_limit"]:.2f}')
    print(f'  50% size limit   : {stats["r_half_50_limit"]:.2f} px')
    print()
    print('=== Photometric Accuracy ===')
    print(f'  Mean mag offset  : {stats["mean_mag_offset"]:+.3f} mag')
    print(f'  Std  mag offset  : {stats["std_mag_offset"]:.3f} mag')

In [ ]:
# ---- Completeness vs Magnitude -----------------------------------------------
if len(detections) > 0:
    mag_bins = np.linspace(config.cluster_config.mag_min,
                           config.cluster_config.mag_max, 15)

    bin_centers_mag, comp_mag, comp_mag_err = retrieval.compute_completeness(
        parameter='magnitude', bins=mag_bins
    )
    mag_50 = retrieval.get_50_percent_limit('magnitude')

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.step(bin_centers_mag, comp_mag, where='mid', lw=2, color='royalblue',
            label='Completeness')
    ax.fill_between(bin_centers_mag,
                    comp_mag - comp_mag_err,
                    comp_mag + comp_mag_err,
                    step='mid', alpha=0.3, color='royalblue',
                    label='1-sigma binomial error')
    ax.axhline(0.5, ls='--', color='gray', lw=1.2, label='50%')
    ax.axhline(0.9, ls=':',  color='gray', lw=1.2, label='90%')
    if not np.isnan(mag_50):
        ax.axvline(mag_50, ls='--', color='tomato', lw=1.5,
                   label=f'50% limit = {mag_50:.2f} mag')
    ax.set_xlabel('Injected Magnitude',  fontsize=13)
    ax.set_ylabel('Completeness',        fontsize=13)
    ax.set_ylim(-0.05, 1.15)
    ax.set_title('Completeness vs Magnitude', fontsize=14)
    ax.legend(fontsize=10)
    plt.tight_layout()
    plt.savefig(os.path.join(config.output_dir, 'completeness_vs_mag.png'), dpi=150)
    plt.show()
    print(f'50% completeness magnitude limit: {mag_50:.2f}')
else:
    print('Skipping completeness vs magnitude (no detections).')

In [ ]:
# ---- Completeness vs Half-light Radius ---------------------------------------
if len(detections) > 0:
    rh_bins = np.linspace(config.cluster_config.r_half_min,
                          config.cluster_config.r_half_max, 10)

    bin_centers_rh, comp_rh, comp_rh_err = retrieval.compute_completeness(
        parameter='r_half', bins=rh_bins
    )
    rh_50 = retrieval.get_50_percent_limit('r_half')

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.step(bin_centers_rh, comp_rh, where='mid', lw=2, color='darkorange',
            label='Completeness')
    ax.fill_between(bin_centers_rh,
                    comp_rh - comp_rh_err,
                    comp_rh + comp_rh_err,
                    step='mid', alpha=0.3, color='darkorange',
                    label='1-sigma binomial error')
    ax.axhline(0.5, ls='--', color='gray', lw=1.2, label='50%')
    ax.axhline(0.9, ls=':',  color='gray', lw=1.2, label='90%')
    if not np.isnan(rh_50):
        ax.axvline(rh_50, ls='--', color='tomato', lw=1.5,
                   label=f'50% limit = {rh_50:.2f} px')
    ax.set_xlabel('Half-light Radius (pixels)', fontsize=13)
    ax.set_ylabel('Completeness',               fontsize=13)
    ax.set_ylim(-0.05, 1.15)
    ax.set_title('Completeness vs Half-light Radius', fontsize=14)
    ax.legend(fontsize=10)
    plt.tight_layout()
    plt.savefig(os.path.join(config.output_dir, 'completeness_vs_rh.png'), dpi=150)
    plt.show()
    print(f'50% completeness half-light radius limit: {rh_50:.2f} px')
else:
    print('Skipping completeness vs size (no detections).')

In [ ]:
# ---- Completeness vs Concentration -------------------------------------------
if len(detections) > 0:
    conc_bins = np.linspace(config.cluster_config.concentration_min,
                            config.cluster_config.concentration_max, 8)

    bin_centers_conc, comp_conc, comp_conc_err = retrieval.compute_completeness(
        parameter='concentration', bins=conc_bins
    )
    conc_50 = retrieval.get_50_percent_limit('concentration')

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.step(bin_centers_conc, comp_conc, where='mid', lw=2,
            color='seagreen', label='Completeness')
    ax.fill_between(bin_centers_conc,
                    comp_conc - comp_conc_err,
                    comp_conc + comp_conc_err,
                    step='mid', alpha=0.3, color='seagreen',
                    label='1-sigma binomial error')
    ax.axhline(0.5, ls='--', color='gray', lw=1.2, label='50%')
    ax.axhline(0.9, ls=':',  color='gray', lw=1.2, label='90%')
    if not np.isnan(conc_50):
        ax.axvline(conc_50, ls='--', color='tomato', lw=1.5,
                   label=f'50% limit = {conc_50:.1f}')
    ax.set_xlabel('Concentration (r_tidal / r_core)', fontsize=13)
    ax.set_ylabel('Completeness',                      fontsize=13)
    ax.set_ylim(-0.05, 1.15)
    ax.set_title('Completeness vs Concentration', fontsize=14)
    ax.legend(fontsize=10)
    plt.tight_layout()
    plt.savefig(os.path.join(config.output_dir, 'completeness_vs_conc.png'), dpi=150)
    plt.show()
else:
    print('Skipping completeness vs concentration (no detections).')

In [ ]:
# ---- 2D Completeness Map (Magnitude x Half-light Radius) --------------------
if len(detections) > 0:
    mag_bins_2d = np.linspace(config.cluster_config.mag_min,
                              config.cluster_config.mag_max, 10)
    rh_bins_2d  = np.linspace(config.cluster_config.r_half_min,
                               config.cluster_config.r_half_max, 8)

    result_2d = retrieval.compute_completeness_2d(
        param1='magnitude', param2='r_half',
        bins1=mag_bins_2d,  bins2=rh_bins_2d,
    )
    comp_2d  = result_2d['completeness']
    bc_mag   = result_2d['bin_centers1']
    bc_rh    = result_2d['bin_centers2']
    n_inj_2d = result_2d['n_injected']

    fig, axes = plt.subplots(1, 2, figsize=(15, 5))

    im = axes[0].pcolormesh(mag_bins_2d, rh_bins_2d, comp_2d.T,
                             cmap='RdYlGn', vmin=0, vmax=1)
    plt.colorbar(im, ax=axes[0], label='Completeness')
    for i, mg in enumerate(bc_mag):
        for j, rh in enumerate(bc_rh):
            val = comp_2d[i, j]
            if not np.isnan(val):
                axes[0].text(mg, rh, f'{val:.0%}',
                             ha='center', va='center', fontsize=7,
                             color='black' if 0.2 < val < 0.8 else 'white')
    axes[0].set_xlabel('Injected Magnitude',        fontsize=12)
    axes[0].set_ylabel('Half-light Radius (pixels)', fontsize=12)
    axes[0].set_title('2D Completeness Map',         fontsize=13)

    im2 = axes[1].pcolormesh(mag_bins_2d, rh_bins_2d, n_inj_2d.T, cmap='Blues')
    plt.colorbar(im2, ax=axes[1], label='N injected')
    for i, mg in enumerate(bc_mag):
        for j, rh in enumerate(bc_rh):
            n = int(n_inj_2d[i, j])
            if n > 0:
                axes[1].text(mg, rh, str(n),
                             ha='center', va='center', fontsize=7)
    axes[1].set_xlabel('Injected Magnitude',        fontsize=12)
    axes[1].set_ylabel('Half-light Radius (pixels)', fontsize=12)
    axes[1].set_title('N Injected per Cell',         fontsize=13)

    plt.suptitle(
        f'Completeness Analysis -- {config.cluster_config.profile_type.upper()} profile',
        fontsize=14,
    )
    plt.tight_layout()
    plt.savefig(os.path.join(config.output_dir, 'completeness_2d.png'), dpi=150)
    plt.show()
else:
    print('Skipping 2D completeness map (no detections).')

---
## 11. Save Results

In [ ]:
# Attach results to the pipeline object for save_results()
pipe.injected_image    = injected_image
pipe.injection_info    = injection_info
pipe.detection_catalog = detections
if len(detections) > 0:
    pipe.retrieval = retrieval

try:
    pipe.save_results()
    print(f'All outputs saved to: {config.output_dir}')
except Exception as e:
    print(f'save_results issue: {e}')
    import pandas as pd
    df = pd.DataFrame(injection_info)
    df.drop(columns=['stamp'], errors='ignore', inplace=True)
    outpath = os.path.join(config.output_dir, 'injection_catalog.csv')
    df.to_csv(outpath, index=False)
    print(f'Saved injection catalog to {outpath}')

print()
print('=== Pipeline complete ===')
print(f'Outputs in          : {config.output_dir}/')
print(f'Injected clusters   : {len(injection_info)}')
print(f'Detections          : {len(detections)}')

---
## 12. Poster Exports (Large-Text Figures)

This section adds poster-ready outputs at the end of the full Rubin PSF pipeline:
- large-text completeness plots
- large-text original/injected/difference panel
- a couple of large-text postage stamp triptychs

In [ ]:
# ---- Poster plotting settings + output folder --------------------------------
poster_dir = os.path.join(config.output_dir, 'poster_exports')
os.makedirs(poster_dir, exist_ok=True)

poster_style = {
    'figure.dpi': 140,
    'savefig.dpi': 350,
    'font.size': 20,
    'axes.titlesize': 28,
    'axes.labelsize': 22,
    'xtick.labelsize': 18,
    'ytick.labelsize': 18,
    'legend.fontsize': 16,
}

print(f'Poster outputs will be written to: {poster_dir}')

In [ ]:
# ---- Poster exports: completeness + difference + postage stamps ---------------
from matplotlib.colors import SymLogNorm

with plt.rc_context(poster_style):
    # Build retrieval if needed (same logic as pipeline section)
    if len(detections) > 0 and 'retrieval' not in globals():
        retrieval = ClusterRetrieval(injection_info, detections)
        retrieval.match_detections(match_radius=MATCH_RADIUS)

    # 1) Large-text completeness plots
    if len(detections) > 0:
        mag_bins = np.linspace(config.cluster_config.mag_min, config.cluster_config.mag_max, 15)
        bc_mag, comp_mag, comp_mag_err = retrieval.compute_completeness('magnitude', mag_bins)
        mag_50 = retrieval.get_50_percent_limit('magnitude')

        fig, ax = plt.subplots(figsize=(11, 7))
        ax.step(bc_mag, comp_mag, where='mid', lw=3, color='royalblue', label='Completeness')
        ax.fill_between(bc_mag, comp_mag - comp_mag_err, comp_mag + comp_mag_err,
                        step='mid', alpha=0.25, color='royalblue', label='1-sigma')
        ax.axhline(0.5, ls='--', color='gray', lw=2)
        if not np.isnan(mag_50):
            ax.axvline(mag_50, ls='--', color='tomato', lw=2.5, label=f'50% = {mag_50:.2f} mag')
        ax.set_xlabel('Injected Magnitude', fontweight='bold')
        ax.set_ylabel('Completeness', fontweight='bold')
        ax.set_ylim(-0.05, 1.05)
        ax.set_title('Completeness vs Magnitude', fontweight='bold')
        ax.grid(alpha=0.3)
        ax.legend(loc='best')
        plt.tight_layout()
        out_mag = os.path.join(poster_dir, 'poster_completeness_vs_mag.png')
        plt.savefig(out_mag, bbox_inches='tight')
        plt.show()

        rh_bins = np.linspace(config.cluster_config.r_half_min, config.cluster_config.r_half_max, 10)
        bc_rh, comp_rh, comp_rh_err = retrieval.compute_completeness('r_half', rh_bins)
        rh_50 = retrieval.get_50_percent_limit('r_half')

        fig, ax = plt.subplots(figsize=(11, 7))
        ax.step(bc_rh, comp_rh, where='mid', lw=3, color='darkorange', label='Completeness')
        ax.fill_between(bc_rh, comp_rh - comp_rh_err, comp_rh + comp_rh_err,
                        step='mid', alpha=0.25, color='darkorange', label='1-sigma')
        ax.axhline(0.5, ls='--', color='gray', lw=2)
        if not np.isnan(rh_50):
            ax.axvline(rh_50, ls='--', color='tomato', lw=2.5, label=f'50% = {rh_50:.2f} px')
        ax.set_xlabel('Half-light Radius (pixels)', fontweight='bold')
        ax.set_ylabel('Completeness', fontweight='bold')
        ax.set_ylim(-0.05, 1.05)
        ax.set_title('Completeness vs Half-light Radius', fontweight='bold')
        ax.grid(alpha=0.3)
        ax.legend(loc='best')
        plt.tight_layout()
        out_rh = os.path.join(poster_dir, 'poster_completeness_vs_rh.png')
        plt.savefig(out_rh, bbox_inches='tight')
        plt.show()

        print('Saved completeness poster plots:')
        print(' -', out_mag)
        print(' -', out_rh)
    else:
        print('Skipping poster completeness plots (no detections).')

    # 2) Large-text full-frame original/injected/difference panel
    vmin, vmax = np.percentile(image, [1, 99])
    diff = injected_image.astype(np.float64) - image.astype(np.float64)
    dmax = max(np.nanpercentile(np.abs(diff), 99.5), 1e-6)

    fig, axes = plt.subplots(1, 3, figsize=(24, 8), constrained_layout=True)

    im0 = axes[0].imshow(image, cmap='gray', origin='lower', vmin=vmin, vmax=vmax)
    axes[0].set_title('Original Image', fontweight='bold')
    axes[0].set_xlabel('X [pix]', fontweight='bold')
    axes[0].set_ylabel('Y [pix]', fontweight='bold')
    c0 = fig.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.02)
    c0.set_label('Flux', fontweight='bold')

    im1 = axes[1].imshow(injected_image, cmap='gray', origin='lower', vmin=vmin, vmax=vmax)
    axes[1].set_title('Injected Image', fontweight='bold')
    axes[1].set_xlabel('X [pix]', fontweight='bold')
    axes[1].set_ylabel('Y [pix]', fontweight='bold')
    c1 = fig.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.02)
    c1.set_label('Flux', fontweight='bold')

    im2 = axes[2].imshow(
        diff,
        cmap='coolwarm',
        origin='lower',
        norm=SymLogNorm(linthresh=max(dmax / 30, 1e-6), vmin=-dmax, vmax=dmax),
    )
    axes[2].set_title('Difference (Injected - Original)', fontweight='bold')
    axes[2].set_xlabel('X [pix]', fontweight='bold')
    axes[2].set_ylabel('Y [pix]', fontweight='bold')
    c2 = fig.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.02)
    c2.set_label('Delta Flux', fontweight='bold')

    fig.suptitle('Injection Diagnostics (Poster View)', fontsize=32, fontweight='bold')
    out_full = os.path.join(poster_dir, 'poster_original_injected_difference.png')
    plt.savefig(out_full, bbox_inches='tight')
    plt.show()
    print('Saved:', out_full)

    # 3) Two large postage-stamp triptychs
    stamp_half = 28
    n_show = min(2, len(injection_info))

    for i in range(n_show):
        info = injection_info[i]
        x0 = int(round(info['x']))
        y0 = int(round(info['y']))

        y_min = max(0, y0 - stamp_half)
        y_max = min(image.shape[0], y0 + stamp_half + 1)
        x_min = max(0, x0 - stamp_half)
        x_max = min(image.shape[1], x0 + stamp_half + 1)

        s0 = image[y_min:y_max, x_min:x_max]
        s1 = injected_image[y_min:y_max, x_min:x_max]
        sd = s1.astype(np.float64) - s0.astype(np.float64)

        svmin, svmax = np.percentile(np.concatenate([s0.ravel(), s1.ravel()]), [1, 99])
        sdmax = max(np.nanpercentile(np.abs(sd), 99.5), 1e-6)

        fig, axs = plt.subplots(1, 3, figsize=(18, 6), constrained_layout=True)

        ims0 = axs[0].imshow(s0, cmap='gray', origin='lower', vmin=svmin, vmax=svmax)
        axs[0].set_title('Original Stamp', fontweight='bold')
        axs[0].set_xlabel('X [pix]', fontweight='bold')
        axs[0].set_ylabel('Y [pix]', fontweight='bold')
        fig.colorbar(ims0, ax=axs[0], fraction=0.046, pad=0.02)

        ims1 = axs[1].imshow(s1, cmap='gray', origin='lower', vmin=svmin, vmax=svmax)
        axs[1].set_title('Injected Stamp', fontweight='bold')
        axs[1].set_xlabel('X [pix]', fontweight='bold')
        axs[1].set_ylabel('Y [pix]', fontweight='bold')
        fig.colorbar(ims1, ax=axs[1], fraction=0.046, pad=0.02)

        ims2 = axs[2].imshow(
            sd,
            cmap='coolwarm',
            origin='lower',
            norm=SymLogNorm(linthresh=max(sdmax / 30, 1e-6), vmin=-sdmax, vmax=sdmax),
        )
        axs[2].set_title('Difference (Injected - Original)', fontweight='bold')
        axs[2].set_xlabel('X [pix]', fontweight='bold')
        axs[2].set_ylabel('Y [pix]', fontweight='bold')
        fig.colorbar(ims2, ax=axs[2], fraction=0.046, pad=0.02)

        fig.suptitle(
            f'Poster Stamp #{i + 1}: (x, y)=({info["x"]:.1f}, {info["y"]:.1f}), m={info["magnitude"]:.2f}',
            fontsize=26,
            fontweight='bold',
        )
        out_stamp = os.path.join(poster_dir, f'poster_stamp_triptych_{i + 1:02d}.png')
        plt.savefig(out_stamp, bbox_inches='tight')
        plt.show()
        print('Saved:', out_stamp)